In [ ]:
import os
import json
import math
import matplotlib.pyplot as plt
import matplotlib.image as mpimg


def create_figure_grid(
    base_directory,
    figure_rel_path="figures/discrepancy_diagnostics_realspace.png",
    config_filename="used_config.json",
    text_keys=[
        ("calibration_settings", "kappa_prior_vals"),
        ("calibration_settings", "delta_eta_prior_vals"),
    ],
    ncols=4,
    figsize=(16, 12),
    output_path="grid_summary.png"
):
    """
    Create a grid of figures from config directories with overlaid text.

    Parameters
    ----------
    base_directory : str
        Path containing config_XXXX directories

    figure_rel_path : str
        Relative path inside each config directory to the figure

    config_filename : str
        Name of JSON file containing configuration

    text_keys : list of tuples
        Keys to extract from JSON in hierarchical form
        Example: [("calibration_settings", "kappa_prior_vals")]

    ncols : int
        Number of columns in grid

    figsize : tuple
        Size of the full figure

    output_path : str
        Path to save the resulting grid image
    """

    # --- find config directories ---
    config_dirs = sorted([
        d for d in os.listdir(base_directory)
        if d.startswith("config_") and os.path.isdir(os.path.join(base_directory, d))
    ])

    images = []
    labels = []

    for config_dir in config_dirs:
        full_dir = os.path.join(base_directory, config_dir)

        fig_path = os.path.join(full_dir, figure_rel_path)
        config_path = os.path.join(full_dir, config_filename)

        # --- skip missing files ---
        if not os.path.exists(fig_path) or not os.path.exists(config_path):
            continue

        # --- load image ---
        img = mpimg.imread(fig_path)

        # --- load config ---
        with open(config_path, "r") as f:
            config = json.load(f)

        # --- extract text ---
        text_lines = [config_dir]

        for key_tuple in text_keys:
            val = config
            try:
                for k in key_tuple:
                    val = val[k]
                text_lines.append(f"{key_tuple[-1]}: {val}")
            except KeyError:
                text_lines.append(f"{key_tuple[-1]}: N/A")

        label_text = "\n".join(text_lines)

        images.append(img)
        labels.append(label_text)

    # --- grid layout ---
    n_images = len(images)
    nrows = math.ceil(n_images / ncols)

    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)

    # flatten axes safely
    axes = axes.flatten() if n_images > 1 else [axes]

    for i, ax in enumerate(axes):
        if i < n_images:
            ax.imshow(images[i])
            ax.axis("off")

            # overlay text
            ax.text(
                0.02, 0.98,
                labels[i],
                transform=ax.transAxes,
                fontsize=8,
                verticalalignment='top',
                bbox=dict(facecolor='white', alpha=0.7, edgecolor='none')
            )
        else:
            ax.axis("off")

    plt.tight_layout()
    plt.savefig(output_path, dpi=200)
    plt.close()

    print(f"Saved grid figure to {output_path}")

In [4]:
# plotting function for creating the physical space plot
def plot_discrepancy_diagnostics(
    x_obs,
    y_obs,
    x_sim,
    y_sim,
    y_prior_mean,
    y_prior_var,
    delta_eta_mean,
    delta_eta_std,
    y_post_mean,
    y_post_var,
    theta_fixed,
    theta_fixed_phys,
    gp_eta,
    kappa_mean,
    kappa_std,
    idx,
    dtheta,
    cross_validation_settings,
    figures_directory=None,
    figure_name="discrepancy_diagnostics.png",
    suptitle="Discrepancy Diagnostics",
):
    x = np.asarray(x_obs).ravel()
    y_obs = np.asarray(y_obs).ravel()
    y_prior_mean = np.asarray(y_prior_mean).ravel()
    y_prior_var = np.asarray(y_prior_var).ravel()
    delta_eta_mean = np.asarray(delta_eta_mean).ravel()
    delta_eta_std = np.asarray(delta_eta_std).ravel()
    y_post_mean = np.asarray(y_post_mean).ravel()
    y_post_var = np.asarray(y_post_var).ravel()

    # If prior arrays are not on x_obs, recompute at x_obs using gp_eta + theta_fixed
    if y_prior_mean.shape[0] != x.shape[0] or y_prior_var.shape[0] != x.shape[0]:
        Z_obs = np.hstack([x.reshape(-1, 1), np.tile(theta_fixed, (x.shape[0], 1))])
        y_prior_mean = gp_eta.predict(Z_obs)
        y_prior_var = np.diag(gp_eta.predict(Z_obs, return_cov=True)[1])

    # Layout:
    # row 0 -> 2 columns (ax1, ax2)
    # rows 1..dtheta -> one full-width subplot each for delta_theta[k]
    # last row -> one full-width subplot for posterior
    n_rows = 2 + dtheta
    fig = plt.figure(figsize=(10, 3.2 * n_rows))
    gs = fig.add_gridspec(n_rows, 2)

    # Top Left: Emulator Prior
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.scatter(x, y_obs, label="Observed", color="black")
    ax1.plot(x, y_prior_mean, label="η(x, θ)", linestyle="--")
    ax1.scatter(np.asarray(x_sim).ravel(), np.asarray(y_sim).ravel(),
                label="Simulator Data", color="blue", alpha=0.5)
    # ax1.fill_between(
    #     x,
    #     y_prior_mean - 2 * np.sqrt(y_prior_var),
    #     y_prior_mean + 2 * np.sqrt(y_prior_var),
    #     alpha=0.3
    # )
    ax1.set_title("Emulator Prior (No Discrepancy)")
    ax1.set_xlabel("x")
    ax1.set_ylabel("y")
    ax1.legend()

    # Top Right: δ_eta contribution
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.plot(x, delta_eta_mean, label="$\delta_\eta$ mean")
    ax2.fill_between(
        x,
        delta_eta_mean - 2 * delta_eta_std,
        delta_eta_mean + 2 * delta_eta_std,
        alpha=0.3
    )
    if cross_validation_settings['conduct_cross_validation'] is True and "known_delta_form" in cross_validation_settings:
        known_delta_form = cross_validation_settings.get("known_delta_form")
        known_delta_form_params = cross_validation_settings.get("known_delta_form_params", {})
        form_config = known_delta_form_params.get(known_delta_form, {})
        
        if known_delta_form == "power_law" and form_config:
            coeff = form_config.get("coeff")
            exponent = form_config.get("exponent")
            if coeff is not None and exponent is not None:
                delta_known = coeff * np.power(x, exponent)
                ax2.plot(x, delta_known, label=r"$\delta_\eta^{\mathrm{true}}(x)$", linestyle="--", color="red")
        elif known_delta_form == "trig_funct" and form_config:
            trig_function = form_config.get("function")
            if trig_function == "sin":
                delta_known = np.sin(x)
            elif trig_function == "cos":
                delta_known = np.cos(x)
            else:
                delta_known = np.zeros_like(x)
            ax2.plot(x, delta_known, label=r"$\delta_\eta^{\mathrm{true}}(x)$", linestyle="--", color="red")
        
    ax2.axhline(0, linestyle="--")
    ax2.set_title(r"Additive Discrepancy $\delta_{\eta}(x)$")
    ax2.set_xlabel("x")
    ax2.set_ylabel("Correction")
    ax2.legend()

    # Following subplots in a single column (full width)
    for k in range(dtheta):
        ax_k = fig.add_subplot(gs[1 + k, :])
        # delta_theta_k_mean = delta_theta_chain[idx, k, :].mean(axis=0)
        # delta_theta_k_std = delta_theta_chain[idx, k, :].std(axis=0)

        delta_theta_k_mean = kappa_mean[k, :]
        delta_theta_k_std  = kappa_std[k, :]

        ax_k.plot(x, delta_theta_k_mean, label=f"\kappa_{k} mean")
        ax_k.fill_between(
            x,
            delta_theta_k_mean - 2 * delta_theta_k_std,
            delta_theta_k_mean + 2 * delta_theta_k_std,
            alpha=0.3
        )
        # Fix previously-added curve label formatting
        if ax_k.lines:
            ax_k.lines[-1].set_label(rf"$\kappa_{{{k}}}(x)$ mean")
            
        if cross_validation_settings['conduct_cross_validation'] is True and "known_theta_form" in cross_validation_settings:
            known_theta_form = cross_validation_settings.get("known_theta_form")
            known_theta_form_params = cross_validation_settings.get("known_theta_form_params", {})
            form_config = known_theta_form_params.get(known_theta_form, {})

            if known_theta_form == "constant" and "values" in form_config:
                known_theta_values = form_config.get("values")
                if k < len(known_theta_values):
                    theta_known = known_theta_values[k] - theta_fixed_phys[k]
                    ax_k.axhline(theta_known, linestyle="--", color="red", label=rf"$\kappa_{{{k}}}^{{\mathrm{{true}}}}$")
            elif known_theta_form == "trig_funct" and "functions" in form_config:
                trig_functions = form_config.get("functions")
                if k < len(trig_functions):
                    func_name = trig_functions[k]
                    if func_name == "sin":
                        theta_known = np.sin(x) - theta_fixed_phys[k]
                    elif func_name == "cos":
                        theta_known = np.cos(x) - theta_fixed_phys[k]
                    else:
                        theta_known = np.zeros_like(x)
                ax_k.plot(x, theta_known, linestyle="--", color="red", label=rf"$\kappa_{{{k}}}^{{\mathrm{{true}}}}(x)$")

        ax_k.axhline(0, linestyle="--", color="gray", label=r"$\theta_0$")
        ax_k.set_title(rf"Calibration discrepancy: $\kappa_{{{k}}}(x)$")
        ax_k.set_xlabel("x")
        ax_k.set_ylabel(rf"$\kappa_{{{k}}}(x)$")
        ax_k.legend()

    # Bottom: Full Posterior (also full width)
    ax3 = fig.add_subplot(gs[1 + dtheta, :])
    ax3.scatter(x, y_obs, label="Observed", color="black")
    ax3.plot(x, y_post_mean, label="Posterior Mean")
    ax3.fill_between(
        x,
        y_post_mean - 2 * np.sqrt(y_post_var),
        y_post_mean + 2 * np.sqrt(y_post_var),
        alpha=0.3
    )
    ax3.set_title("Full Posterior Prediction")
    ax3.set_xlabel("x")
    ax3.set_ylabel("y")
    ax3.legend()

    plt.suptitle(suptitle)
    plt.tight_layout()
    # plt.show()
    plt_path = os.path.join(figures_directory, figure_name)
    plt.savefig(plt_path, dpi=150)
    

    return()

<>:67: SyntaxWarning: invalid escape sequence '\d'
<>:110: SyntaxWarning: invalid escape sequence '\k'
<>:67: SyntaxWarning: invalid escape sequence '\d'
<>:110: SyntaxWarning: invalid escape sequence '\k'
/tmp/ipykernel_41661/1384695606.py:67: SyntaxWarning: invalid escape sequence '\d'
  ax2.plot(x, delta_eta_mean, label="$\delta_\eta$ mean")
/tmp/ipykernel_41661/1384695606.py:110: SyntaxWarning: invalid escape sequence '\k'
  ax_k.plot(x, delta_theta_k_mean, label=f"\kappa_{k} mean")


In [7]:
import os
import numpy as np
# read the current set of results from box
base_dir = "/home/liammyhill/Desktop/box/calibrationResults/fuid/paper_finalRuns"

# select the runs that we want to re-plot
study1 = {
    "ell_k": 10,
    "sigma_k": 0.1,
    "ell_delta": 0.1,
    "sigma_delta": 0.1
}
study2 = {
    "ell_k": 0.5,
    "sigma_k": 0.2,
    "ell_delta": 0.1,
    "sigma_delta": 0.1
}

# Build paths based on the parameter dictionaries
# Format: kvar{sigma_k}_kell{ell_k}_evar{sigma_delta}_eell{ell_delta}

study1_dir = os.path.join(
    base_dir, 
    "study1/results/sweep",
    f"kvar{study1['sigma_k']}_kell{study1['ell_k']}_evar{study1['sigma_delta']}_eell{study1['ell_delta']}"
)

study2_dir = os.path.join(
    base_dir,
    "study2/results/sweep", 
    f"kvar{study2['sigma_k']}_kell{study2['ell_k']}_evar{study2['sigma_delta']}_eell{study2['ell_delta']}"
)

print(f"Study 1 path: {study1_dir}")
print(f"Study 2 path: {study2_dir}")

# from each directory, load results_physical.npz
study1_results_path = os.path.join(study1_dir, "results_physical.npz")
study2_results_path = os.path.join(study2_dir, "results_physical.npz")

study1_results = np.load(study1_results_path, allow_pickle=True)
study2_results = np.load(study2_results_path, allow_pickle=True)

#print the headers of the loaded results to check what keys are available
print("Study 1 results keys:", study1_results.files)
print("Study 2 results keys:", study2_results.files)

#plot the results using the plotting function defined above
plot_discrepancy_diagnostics(
    x_obs=study1_results["x_obs"],
    y_obs=study1_results["y_obs"],
    x_sim=study1_results["x_sim"],
    y_sim=study1_results["y_sim"],
    y_prior_mean=study1_results["y_prior_mean"],
    y_prior_var=study1_results["y_prior_var"],
    delta_eta_mean=study1_results["delta_eta_mean"],
    delta_eta_std=study1_results["delta_eta_std"],
    y_post_mean=study1_results["y_post_mean"],
    y_post_var=study1_results["y_post_var"],
    theta_fixed=study1_results["theta_fixed"],
    theta_fixed_phys=study1_results["theta_fixed_phys"],
    gp_eta=study1_results["gp_eta"].item(),
    kappa_mean=study1_results["kappa_mean"],
    kappa_std=study1_results["kappa_std"],
    idx=-1,
    dtheta=2,
    cross_validation_settings={
        "conduct_cross_validation": False
    },
    figures_directory=study1_dir,
    figure_name="discrepancy_diagnostics_physical_study1.png",
    suptitle="Discrepancy Diagnostics - Study 1"
)




Study 1 path: /home/liammyhill/Desktop/box/calibrationResults/fuid/paper_finalRuns/study1/results/sweep/kvar0.1_kell10_evar0.1_eell0.1
Study 2 path: /home/liammyhill/Desktop/box/calibrationResults/fuid/paper_finalRuns/study2/results/sweep/kvar0.2_kell0.5_evar0.1_eell0.1
Study 1 results keys: ['x_obs_sim', 'zeta_obs_obs_y', 'y_post_mean', 'y_post_std', 'delta_eta_mean', 'delta_eta_std', 'kappa_0_mean', 'kappa_0_std', 'kappa_1_mean', 'kappa_1_std']
Study 2 results keys: ['x_obs_sim', 'zeta_obs_obs_y', 'y_post_mean', 'y_post_std', 'delta_eta_mean', 'delta_eta_std', 'kappa_0_mean', 'kappa_0_std', 'kappa_1_mean', 'kappa_1_std']


KeyError: 'x_obs is not a file in the archive'